In [1]:
from datasets import load_dataset
languages = ["da_dk", "en_us", "it_it", "de_de"]
dataset = {}

for lang in languages:
   dataset[lang] = load_dataset('rasgaard/fleurs_test', lang)

In [2]:
import copy
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

model_id = "openai/whisper-large-v3-turbo"
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForSpeechSeq2Seq.from_pretrained(model_id)


def prune_encoder_layers(model, layers_to_remove: list[int]):
    pruned = copy.deepcopy(model)
    layers = pruned.model.encoder.layers
    # Remove in reverse order so indices stay valid
    for idx in sorted(set(layers_to_remove), reverse=True):
        del layers[idx]
    return pruned


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

In [3]:
sample = dataset["da_dk"]["train"][0]

In [4]:
import torch

inputs = processor(sample["audio"]["array"], 
                   sampling_rate=sample["audio"]["sampling_rate"], 
                   return_tensors="pt").to(dtype=model.dtype)

with torch.no_grad():
    encoder_outputs = model.model.encoder(inputs.input_features)

hidden_states = encoder_outputs.last_hidden_state
hidden_states

KeyboardInterrupt: 